In [ ]:
%pip install unidecode

In [ ]:
import pandas as pd
import unidecode
import seaborn as sns

In [ ]:
valid_days = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado"]
get_valid_day = lambda day: next(filter(lambda valid_day: unidecode.unidecode(day.strip()).startswith(unidecode.unidecode(valid_day)), valid_days), None)
is_valid_day = lambda day: any(filter(lambda valid_day: unidecode.unidecode(day.strip()).startswith(unidecode.unidecode(valid_day)), valid_days))

df = pd.read_csv("../data/evaluaciones_docentes_sintetica_ampliada.csv", sep=",", encoding="utf-8", header="infer", dtype={
    "id_docente": "category",
    "tendencia_desempeno": "category",
    "comentario": "category",
    "asignatura": "category",
    "franja_horaria": "category",
    "id_curso_grupo": "category",
    "clases_dictadas": "Int64",
    "clases_pactadas": "Int64",
    "numero_estudiantes_aprobaron": "Int64",
    "numero_estudiantes_perdieron": "Int64",
    "numero_estudiantes_desistieron": "Int64",
    "promedio_asistencias_estudiantes": "Float64",
}, converters={
    # Hay semestres donde el docente deja de dictar una materia y necesita agregarse el semestre artificialmente.
    # Por tanto, se agrega esta transformación para más adelante apoyar a la completacion de ceros en semestres faltantes.
    "semestre": lambda s: pd.Period(s.strip().replace("-1", "Q1").replace("-2", "Q3"), freq="2Q-DEC"),
    "hora_inicio": lambda val: pd.to_datetime(val, format="%H:%M").time(),
    "hora_fin": lambda val: pd.to_datetime(val, format="%H:%M").time(),
    "dias_clase": lambda dias: tuple(get_valid_day(dia) for dia in dias.split("-") if is_valid_day(dia)),
})
df = df.convert_dtypes()

df["Cantidad de Clases Semanales"] = df["dias_clase"].apply(lambda dias: len(dias)).astype("Int64")
df["Horas Por Clase"] = ((pd.to_timedelta(df["hora_fin"].astype(str)) - pd.to_timedelta(df["hora_inicio"].astype(str))).dt.total_seconds() / 3600).astype("Int64")

# Horas efectivas de operación dirigidas a los estudiantes en el aula
df["Horas Lectivas Totales"] = df["clases_dictadas"] * df["Horas Por Clase"]
df["Horas Lectivas Por Semana"] = df["Cantidad de Clases Semanales"] * df["Horas Por Clase"]

df.dtypes


**Se evidencia que la cantidad de clases pactadas es igual a 16 veces la cantidad de clases semanales, es decir que hay 16 semanas proyectadas para cada grupo.**

**En la práctica quizás reponen una o varias semanas adicionales para garantizar el cumplimiento de horas: bien sea para reponer días compensados del docente, días festivos, situaciones de fuerza mayor, etc.**



In [ ]:
df["Horas Lectivas Teóricas"] = 16 * df["Horas Lectivas Por Semana"]
df["Horas Lectivas Planeadas"] = df["clases_pactadas"] * df["Horas Por Clase"]

(df["Horas Lectivas Teóricas"] == df["Horas Lectivas Planeadas"]).all()



**Validando la relación de horas efectivas y planeadas, se aprecia un 64% de casos en donde se incumplió con lo pactado.**


In [ ]:
porcentaje_casos_incumplimiento = 100 * len(df[df["Horas Lectivas Totales"] < df["Horas Lectivas Planeadas"]]) / len(df)

porcentaje_casos_incumplimiento

**Se aprecia que el 41.4% de los grupos habrían subsanado la brecha de horas con 1 semana extra de clases, ya que corresponden a plazos alcanzables según el horario lectivo.**

**Así mismo, existe un 16.3% de grupos que habrían necesitado dos semanas más para dar cumplimiento a las horas faltantes.**

**Por último, el 6.3% de grupos habrían necesitado mucho más tiempo para cumplir.**


In [ ]:
df["Discrepancia de Horas Lectivas"] = (df["Horas Lectivas Planeadas"] - df["Horas Lectivas Totales"]).astype("Int64")

mask_viabilidad_subsanado_una_semana = (
    (df["Discrepancia de Horas Lectivas"] > 0)
    # Discrepancia NO puede superar las horas máximas de una semana
    & (df["Discrepancia de Horas Lectivas"] <= df["Horas Lectivas Por Semana"])
)

mask_viabilidad_subsanado_dos_semanas = (
    (df["Discrepancia de Horas Lectivas"] > 0)
    & (df["Discrepancia de Horas Lectivas"] > df["Horas Lectivas Por Semana"])
    & (df["Discrepancia de Horas Lectivas"] <= (2 * df["Horas Lectivas Por Semana"]))
)

mask_no_viables = (
    (df["Discrepancia de Horas Lectivas"] > 0)
    & (df["Discrepancia de Horas Lectivas"] > (2 * df["Horas Lectivas Por Semana"]))
)

grupos_una_semana_mas = len(df[mask_viabilidad_subsanado_una_semana])
grupos_dos_semanas_mas = len(df[mask_viabilidad_subsanado_dos_semanas])
grupos_no_viables = len(df[mask_no_viables])

porcentaje_casos_remediables_una_semana = 100 *  grupos_una_semana_mas / len(df)
porcentaje_casos_remediables_dos_semanas = 100 *  grupos_dos_semanas_mas / len(df)
porcentaje_casos_no_viables = 100 *  grupos_no_viables / len(df)

print(
    f"Cantidad de Grupos que habrían necesitado 1 semana mas: {grupos_una_semana_mas } (el {porcentaje_casos_remediables_una_semana:.2f}%)", "\n",
    f"Cantidad de Grupos que habrían necesitado 2 semanas mas: {grupos_dos_semanas_mas } (el {porcentaje_casos_remediables_dos_semanas:.2f}%)", "\n",
    f"Cantidad de Grupos que habrían necesitado mucho mas tiempo: {grupos_no_viables} (el {porcentaje_casos_no_viables:.2f}%)"
)

In [ ]:
seguimiento_general_docentes = (
    df.groupby(["id_docente", "semestre"], as_index=False)
    .agg(
        Cantidad_de_Materias=("asignatura", "nunique"),
        Cantidad_de_Grupos=("semestre", "size"),
        numero_estudiantes=("numero_estudiantes", "sum"),
        Asignaturas=("asignatura", lambda x: tuple(x.dropna().unique())),
        Horas_Lectivas=("Horas Lectivas Totales", "sum"),
        Numero_Comentarios_Estables=("tendencia_desempeno", lambda x: len([tendencia for tendencia in x.dropna() if tendencia == "Estable"])),
        Numero_Comentarios_En_Riesgo=("tendencia_desempeno", lambda x: len([tendencia for tendencia in x.dropna() if tendencia == "En riesgo"])),
        Numero_Comentarios_En_Mejora=("tendencia_desempeno", lambda x: len([tendencia for tendencia in x.dropna() if tendencia == "Mejora"])),
    )
    .rename(columns={
        "Cantidad_de_Grupos": "Cantidad de Grupos",
        "Cantidad_de_Materias": "Cantidad de Materias",
        "Horas_Lectivas": "Horas Lectivas",
        "Numero_Comentarios_Estables": "Numero Comentarios Estables",
        "Numero_Comentarios_En_Riesgo": "Numero Comentarios En Riesgo",
        "Numero_Comentarios_En_Mejora": "Numero Comentarios En Mejora"
    })
    .sort_values(by=["id_docente", "semestre"], ascending=[True, True])
)

seguimiento_general_docentes["Cantidad de Materias"] = seguimiento_general_docentes["Cantidad de Materias"].astype("Int64")
seguimiento_general_docentes["Cantidad de Grupos"] = seguimiento_general_docentes["Cantidad de Grupos"].astype("Int64")
seguimiento_general_docentes["numero_estudiantes"] = seguimiento_general_docentes["numero_estudiantes"].astype("Int64")

seguimiento_general_docentes["Numero Comentarios Estables"] = seguimiento_general_docentes["Numero Comentarios Estables"].astype("Int64")
seguimiento_general_docentes["Numero Comentarios En Riesgo"] = seguimiento_general_docentes["Numero Comentarios En Riesgo"].astype("Int64")
seguimiento_general_docentes["Numero Comentarios En Mejora"] = seguimiento_general_docentes["Numero Comentarios En Mejora"].astype("Int64")

# cantidad_de_materias_que_el_docente_maneja_por_cada_semestre["Comentarios"] = cantidad_de_materias_que_el_docente_maneja_por_cada_semestre["Comentarios"].apply(list)
# cantidad_de_materias_que_el_docente_maneja_por_cada_semestre["Asignaturas"] = cantidad_de_materias_que_el_docente_maneja_por_cada_semestre["Asignaturas"].apply(list)

def agregar_metricas_generales_docente(muestras_docente):
    muestras = muestras_docente.copy(deep=True)
    muestras.reset_index(inplace=True) # drop multi-index

    muestras.reset_index(inplace=True)  # creates 'index' column with original positions
    muestras["Max Acumulado Cantidad de Materias"] = muestras["Cantidad de Materias"].cummax()
    muestras["Max Acumulado Cantidad de Grupos"] = muestras["Cantidad de Grupos"].cummax()
    muestras["Max Acumulado Numero Comentarios En Mejora"] = muestras["Numero Comentarios En Mejora"].cummax()
    muestras["Max Acumulado Horas Lectivas"] = muestras["Horas Lectivas"].cummax()

    # Campos temporales para hacerle seguimiento a la primera occurencia de maximos locales
    mask_materias = (muestras["Cantidad de Materias"] == muestras["Max Acumulado Cantidad de Materias"])
    mask_grupos = (muestras["Cantidad de Grupos"] == muestras["Max Acumulado Cantidad de Grupos"])
    mask_comentarios_en_mejora = (muestras["Numero Comentarios En Mejora"] == muestras["Max Acumulado Numero Comentarios En Mejora"])
    mask_horas_lectivas = (muestras["Horas Lectivas"] == muestras["Max Acumulado Horas Lectivas"])
    muestras["last_max_idx_materias"] = muestras["index"].where(mask_materias).ffill().astype(int)
    muestras["last_max_idx_grupos"] = muestras["index"].where(mask_grupos).ffill().astype(int)
    muestras["last_max_idx_comentarios_en_mejora"] = muestras["index"].where(mask_comentarios_en_mejora).ffill().astype(int)
    muestras["last_max_idx_horas_lectivas"] = muestras["index"].where(mask_horas_lectivas).ffill().astype(int)

    muestras["Semestres sin Sobrecarga de Asignaturas"] = muestras["index"] - muestras["last_max_idx_materias"]
    muestras["Semestres sin Sobrecarga de Grupos"] = muestras["index"] - muestras["last_max_idx_grupos"]
    muestras["Semestres sin Sobrecarga Horaria"] = muestras["index"] - muestras["last_max_idx_horas_lectivas"]

    muestras["Indice de Carga Asignaturas"] = muestras["Cantidad de Materias"] / muestras["Max Acumulado Cantidad de Materias"]
    muestras["Indice de Carga Grupos"] = muestras["Cantidad de Grupos"] / muestras["Max Acumulado Cantidad de Grupos"]
    muestras["Indice de Carga Horaria"] = muestras["Horas Lectivas"] / muestras["Max Acumulado Horas Lectivas"]

    muestras.drop(columns=["index", "Max Acumulado Cantidad de Materias", "Max Acumulado Cantidad de Grupos", "Max Acumulado Numero Comentarios En Mejora", "Max Acumulado Horas Lectivas", "last_max_idx_materias", "last_max_idx_grupos", "last_max_idx_comentarios_en_mejora", "last_max_idx_horas_lectivas"], inplace=True)
    muestras.set_index(["id_docente"], inplace=True)
    return muestras

seguimiento_general_docentes = (
    seguimiento_general_docentes
    .set_index(["id_docente"], append=False)
    .groupby(level=["id_docente"], group_keys=False)
    .apply(agregar_metricas_generales_docente, include_groups=False)
    .reset_index()
)


In [ ]:
seguimiento_general_docentes

In [ ]:
# Función para devolver la representación de un periodo a formato año-semestre (para visualizaciones, resultados finales, etc)
def period_to_string(periodo: pd.Period):
    return f"{periodo.year}-{(periodo.quarter+1)//2}"

# Más adelante para completar semestres, solo sería revisar el número de periodos entre 2 muestras.
def numero_de_periodos_entre_semestres(semestre_final: pd.Period, semestre_inicial: pd.Period):
    if pd.isna(semestre_final) or pd.isna(semestre_inicial):
        return 0
    return 1 + (semestre_final - semestre_inicial).n//2

seguimiento_docente_en_cada_asignatura = (
    df.groupby(["id_docente", "semestre", "asignatura"], as_index=False)
    .agg(
        Cantidad_de_Grupos=("semestre", "size"), # Counts rows per group
        numero_estudiantes=("numero_estudiantes", "sum"),
        puntaje_claridad=("puntaje_claridad", "mean"),
        puntaje_metodologia=("puntaje_metodologia", "mean"),
        puntaje_evaluacion=("puntaje_evaluacion", "mean"),
        Comentarios=("comentario", lambda x: tuple(x.dropna().unique())),
        Horas_Lectivas=("Horas Lectivas Totales", "sum"),
        Estudiantes_Aprobados=("numero_estudiantes_aprobaron", "sum"),
        Estudiantes_Desertados=("numero_estudiantes_desistieron", "sum"),
        Estudiantes_Perdieron=("numero_estudiantes_perdieron", "sum"),
        Numero_Comentarios_Estables=("tendencia_desempeno", lambda x: len([tendencia for tendencia in x.dropna() if tendencia == "Estable"])),
        Numero_Comentarios_En_Riesgo=("tendencia_desempeno", lambda x: len([tendencia for tendencia in x.dropna() if tendencia == "En riesgo"])),
        Numero_Comentarios_En_Mejora=("tendencia_desempeno", lambda x: len([tendencia for tendencia in x.dropna() if tendencia == "Mejora"])),
    )
    .rename(columns={
        "Cantidad_de_Grupos": "Cantidad de Grupos",
        "Horas_Lectivas": "Horas Lectivas",
        "Estudiantes_Aprobados": "Estudiantes Aprobados",
        "Estudiantes_Desertados": "Estudiantes Desertados",
        "Estudiantes_Perdieron": "Estudiantes Perdieron",

        "Numero_Comentarios_Estables": "Numero Comentarios Estables",
        "Numero_Comentarios_En_Riesgo": "Numero Comentarios En Riesgo",
        "Numero_Comentarios_En_Mejora": "Numero Comentarios En Mejora"
    })
    .sort_values(by=["id_docente", "asignatura", "semestre"], ascending=[True, True, True])
)

seguimiento_docente_en_cada_asignatura["Cantidad de Grupos"] = seguimiento_docente_en_cada_asignatura["Cantidad de Grupos"].astype("Int64")
seguimiento_docente_en_cada_asignatura["numero_estudiantes"] = seguimiento_docente_en_cada_asignatura["numero_estudiantes"].astype("Int64")
seguimiento_docente_en_cada_asignatura["Horas Lectivas"] = seguimiento_docente_en_cada_asignatura["Horas Lectivas"].astype("Int64")
seguimiento_docente_en_cada_asignatura["Estudiantes Aprobados"] = seguimiento_docente_en_cada_asignatura["Estudiantes Aprobados"].astype("Int64")

seguimiento_docente_en_cada_asignatura["Numero Comentarios Estables"] = seguimiento_docente_en_cada_asignatura["Numero Comentarios Estables"].astype("Int64")
seguimiento_docente_en_cada_asignatura["Numero Comentarios En Riesgo"] = seguimiento_docente_en_cada_asignatura["Numero Comentarios En Riesgo"].astype("Int64")
seguimiento_docente_en_cada_asignatura["Numero Comentarios En Mejora"] = seguimiento_docente_en_cada_asignatura["Numero Comentarios En Mejora"].astype("Int64")

def completar_semestres(muestras_docente):

    muestras = muestras_docente.copy(deep=True)
    muestras.reset_index(inplace=True)

    # Campos auxiliares para completación de semestres
    muestras["semestre previo"] = muestras.semestre.shift(1)
    muestras["Numero de Periodos entre Semestres"] = muestras.apply(lambda row: numero_de_periodos_entre_semestres(row["semestre"], row["semestre previo"]), axis=1)

    # Util para darle más contexto al área usuaria y a modelos de inteligencia artificial
    muestras["Semestres Desde Ultima Calificación"] = pd.NA
    muestras.iloc[1:, muestras.columns.get_loc("Semestres Desde Ultima Calificación")] = 1

    gaps = muestras[muestras["Numero de Periodos entre Semestres"] > 2].index.tolist()
    if len(gaps) == 0:
        return muestras_docente

    fields = [
        "semestre",
        "Numero de Periodos entre Semestres",
        "semestre previo",
        "Cantidad de Grupos",
        "Semestres Desde Ultima Calificación",
        "numero_estudiantes",
        "puntaje_claridad",
        "puntaje_metodologia",
        "puntaje_evaluacion",

        "Comentarios",
        "Horas Lectivas",
        "Estudiantes Aprobados",
        "Numero Comentarios Estables",
        "Numero Comentarios En Riesgo",
        "Numero Comentarios En Mejora",
    ]

    # Limita hasta cuantos semestres puedo completar hacia atrás, de momento deja hasta 1000.
    step = 1e-3
    for idx_grupito in gaps:
        semestres_faltantes = muestras.loc[idx_grupito]["Numero de Periodos entre Semestres"] - 2
        muestras.loc[idx_grupito, ["Semestres Desde Ultima Calificación"]] = [muestras.loc[idx_grupito]["Numero de Periodos entre Semestres"] - 1]

        idx_semestre = idx_grupito
        for sem_faltante in range(semestres_faltantes):
            # Clonado y sobre-escritura de campos, asignando el 0 en Cantidad de Grupos y fijando la Última Vez Que Fué Calificado.
            muestras.loc[idx_semestre - step] = muestras.loc[idx_semestre].copy()
            muestras.loc[idx_semestre - step, fields] = [
                muestras.loc[idx_semestre].semestre - 1,
                0,
                pd.NaT,
                pd.NA,
                semestres_faltantes - sem_faltante,
                pd.NA,
                pd.NA,
                pd.NA,
                pd.NA,

                pd.NA,
                pd.NA,
                pd.NA,
                pd.NA,
                pd.NA,
                pd.NA,
            ]
            idx_semestre = idx_semestre - step

    # Estos 2 pasos ordenan correctamente lo que se agregó
    muestras.sort_index(inplace=True)
    muestras.reset_index(drop=True, inplace=True)

    # Ya podemos eliminar los campos auxiliares
    muestras.drop(columns=["semestre previo", "Numero de Periodos entre Semestres"], inplace=True)
    muestras.set_index(["id_docente", "asignatura"], inplace=True)

    muestras["Semestres Desde Ultima Calificación"] = muestras["Semestres Desde Ultima Calificación"].astype("Int64")
    muestras["Cantidad de Grupos"] = muestras["Cantidad de Grupos"].astype("Int64")

    return muestras

seguimiento_docente_en_cada_asignatura = (
    seguimiento_docente_en_cada_asignatura
    .set_index(["id_docente", "asignatura"], append=False)
    .groupby(level=["id_docente", "asignatura"], group_keys=False)
    .apply(completar_semestres, include_groups=False)
    .reset_index()
)

def agregar_metricas_docente_por_asignatura(muestras_docente):
    muestras = muestras_docente.copy(deep=True)

    muestras.reset_index(inplace=True)

    muestras["Diferencia en Cantidad de Grupos con Semestre Anterior"] = (
            muestras["Cantidad de Grupos"]
            - muestras["Cantidad de Grupos"].shift(1)
    )

    muestras["Diferencia en Semestres Desde Ultima Calificación"] = (
            muestras["Semestres Desde Ultima Calificación"]
            - muestras["Semestres Desde Ultima Calificación"].shift(1)
    )

    muestras["Reingreso"] = (
            ( ~ muestras["Cantidad de Grupos"].isna() )
            & muestras["Diferencia en Cantidad de Grupos con Semestre Anterior"].isna()
            & (muestras["Diferencia en Semestres Desde Ultima Calificación"] > 0)
    )

    muestras["Reingreso"] = muestras.apply(lambda row: pd.NA if row["Reingreso"] is False and pd.isna(row["Cantidad de Grupos"]) else row["Reingreso"], axis=1)

    reingresos_acumulados = muestras["Reingreso"].cumsum()
    indice_rotacion = reingresos_acumulados / (reingresos_acumulados.index + 1)

    muestras["Indice de Reingreso"] = indice_rotacion
    # muestras["Indice de Permanencia"] = 1 - muestras["Indice de Reingreso"]
    # muestras["Reingresos a La Fecha"] = reingresos_acumulados
    muestras["Egresos a La Fecha"] = reingresos_acumulados.index - reingresos_acumulados
    # muestras["Indice de Egresos"] = muestras["Egresos a La Fecha"] / (reingresos_acumulados.index + 1)

    muestras.drop(columns=["Diferencia en Cantidad de Grupos con Semestre Anterior", "Diferencia en Semestres Desde Ultima Calificación"], inplace=True)

    muestras.set_index(["id_docente", "asignatura"], inplace=True)

    return muestras

def metricas_de_apoyo_para_detectar_reingreso(muestras_docente):
    muestras = muestras_docente.copy(deep=True)
    muestras.reset_index(inplace=True) # drop multi-index

    muestras.reset_index(inplace=True)  # creates 'index' column with original positions

    muestras["Max Acumulado Numero Comentarios En Riesgo"] = muestras["Numero Comentarios En Riesgo"].cummax()
    mask_comentarios_en_riesgo = (muestras["Numero Comentarios En Riesgo"] == muestras["Max Acumulado Numero Comentarios En Riesgo"])
    muestras["last_max_idx_comentarios_en_riesgo"] = muestras["index"].where(mask_comentarios_en_riesgo).ffill().astype(int)
    muestras["Semestres Sin Incidente Comentarios Negativos"] = muestras["index"] - muestras["last_max_idx_comentarios_en_riesgo"]

    muestras["Ultimo Incidente Comentarios Negativos"] = muestras.apply(
        lambda row: muestras.loc[row["last_max_idx_comentarios_en_riesgo"], "Numero Comentarios En Riesgo"],
        axis=1
    )

    muestras.loc[0, "Semestres Sin Incidente Comentarios Negativos"] = pd.NA
    muestras.loc[0, "Ultimo Incidente Comentarios Negativos"] = pd.NA

    muestras["Max Acumulado Estudiantes Aprobados"] = muestras["Estudiantes Aprobados"].cummax()
    mask_estudiantes_aprobados = (muestras["Estudiantes Aprobados"] == muestras["Max Acumulado Estudiantes Aprobados"])
    muestras["last_max_idx_estudiantes_aprobados"] = muestras["index"].where(mask_estudiantes_aprobados).ffill().astype(int)
    muestras["Semestres Sin Mejora en Estudiantes Aprobados"] = muestras["index"] - muestras["last_max_idx_estudiantes_aprobados"]

    muestras["Ultimo Hito Estudiantes Aprobados"] = muestras.apply(
        lambda row: muestras.loc[row["last_max_idx_estudiantes_aprobados"], "Estudiantes Aprobados"],
        axis=1
    )

    muestras.loc[0, "Ultimo Hito Estudiantes Aprobados"] = pd.NA
    muestras.loc[0, "Semestres Sin Mejora en Estudiantes Aprobados"] = pd.NA

    muestras["Max Acumulado Puntaje Evaluación"] = muestras["puntaje_evaluacion"].cummax()
    mask_puntaje_eval = (muestras["puntaje_evaluacion"] == muestras["Max Acumulado Puntaje Evaluación"])
    muestras["last_max_idx_puntaje_eval"] = muestras["index"].where(mask_puntaje_eval).ffill().astype(int)
    muestras["Semestres Sin Mejora en Puntaje Evaluación"] = muestras["index"] - muestras["last_max_idx_puntaje_eval"]

    muestras["Ultimo Hito Puntaje Evaluación"] = muestras.apply(
        lambda row: muestras.loc[row["last_max_idx_puntaje_eval"], "puntaje_evaluacion"],
        axis=1
    )

    muestras.loc[0, "Ultimo Hito Puntaje Evaluación"] = pd.NA
    muestras.loc[0, "Semestres Sin Mejora en Puntaje Evaluación"] = pd.NA

    muestras["Max Acumulado Puntaje Metodología"] = muestras["puntaje_metodologia"].cummax()
    mask_puntaje_metodologia = (muestras["puntaje_metodologia"] == muestras["Max Acumulado Puntaje Metodología"])
    muestras["last_max_idx_puntaje_metod"] = muestras["index"].where(mask_puntaje_metodologia).ffill().astype(int)
    muestras["Semestres Sin Mejora en Puntaje Metodología"] = muestras["index"] - muestras["last_max_idx_puntaje_metod"]

    muestras["Ultimo Hito Puntaje Metodología"] = muestras.apply(
        lambda row: muestras.loc[row["last_max_idx_puntaje_metod"], "puntaje_metodologia"],
        axis=1
    )

    muestras.loc[0, "Ultimo Hito Puntaje Metodología"] = pd.NA
    muestras.loc[0, "Semestres Sin Mejora en Puntaje Metodología"] = pd.NA

    muestras["Max Acumulado Puntaje Claridad"] = muestras["puntaje_claridad"].cummax()
    mask_puntaje_clar = (muestras["puntaje_claridad"] == muestras["Max Acumulado Puntaje Claridad"])
    muestras["last_max_idx_puntaje_clar"] = muestras["index"].where(mask_puntaje_clar).ffill().astype(int)
    muestras["Semestres Sin Mejora en Puntaje Claridad"] = muestras["index"] - muestras["last_max_idx_puntaje_clar"]

    muestras["Ultimo Hito Puntaje Claridad"] = muestras.apply(
        lambda row: muestras.loc[row["last_max_idx_puntaje_clar"], "puntaje_claridad"],
        axis=1
    )

    muestras.loc[0, "Ultimo Hito Puntaje Claridad"] = pd.NA
    muestras.loc[0, "Semestres Sin Mejora en Puntaje Claridad"] = pd.NA


    muestras["Max Acumulado Horas Lectivas"] = muestras["Horas Lectivas"].cummax()
    mask_horas_lectivas = (muestras["Horas Lectivas"] == muestras["Max Acumulado Horas Lectivas"])
    muestras["last_max_idx_horas_lectivas"] = muestras["index"].where(mask_horas_lectivas).ffill().astype(int)
    muestras["Semestres Sin Exceso en Horas Lectivas"] = muestras["index"] - muestras["last_max_idx_horas_lectivas"]

    muestras["Ultimo Tope Horas Lectivas"] = muestras.apply(
        lambda row: muestras.loc[row["last_max_idx_horas_lectivas"], "Horas Lectivas"],
        axis=1
    )

    muestras.loc[0, "Ultimo Tope Horas Lectivas"] = pd.NA
    muestras.loc[0, "Semestres Sin Exceso en Horas Lectivas"] = pd.NA


    muestras["Max Acumulado Estudiantes Desertados"] = muestras["Estudiantes Desertados"].cummax()
    mask_num_est = (muestras["Estudiantes Desertados"] == muestras["Max Acumulado Estudiantes Desertados"])
    muestras["last_max_idx_numero_estudiantes_deser"] = muestras["index"].where(mask_num_est).ffill().astype(int)
    muestras["Semestres Sin Incidente Estudiantes Desertados"] = muestras["index"] - muestras["last_max_idx_numero_estudiantes_deser"]

    muestras["Ultimo Incidente Estudiantes Desertados"] = muestras.apply(
        lambda row: muestras.loc[row["last_max_idx_numero_estudiantes_deser"], "Estudiantes Desertados"],
        axis=1
    )

    muestras.loc[0, "Ultimo Incidente Estudiantes Desertados"] = pd.NA
    muestras.loc[0, "Semestres Sin Incidente Estudiantes Desertados"] = pd.NA


    muestras["Max Acumulado Estudiantes Perdieron"] = muestras["Estudiantes Perdieron"].cummax()
    mask_num_est = (muestras["Estudiantes Perdieron"] == muestras["Max Acumulado Estudiantes Perdieron"])
    muestras["last_max_idx_numero_estudiantes_per"] = muestras["index"].where(mask_num_est).ffill().astype(int)
    muestras["Semestres Sin Incidente Estudiantes Perdieron"] = muestras["index"] - muestras["last_max_idx_numero_estudiantes_per"]

    muestras["Ultimo Incidente Estudiantes Perdieron"] = muestras.apply(
        lambda row: muestras.loc[row["last_max_idx_numero_estudiantes_per"], "Estudiantes Perdieron"],
        axis=1
    )

    muestras.loc[0, "Ultimo Incidente Estudiantes Perdieron"] = pd.NA
    muestras.loc[0, "Semestres Sin Incidente Estudiantes Perdieron"] = pd.NA

    muestras.drop(columns=["index",
                           #"Max Acumulado Numero Comentarios En Mejora",
                           "Max Acumulado Estudiantes Aprobados",
                           "Max Acumulado Puntaje Evaluación",
                           "Max Acumulado Puntaje Metodología",
                           "Max Acumulado Puntaje Claridad",
                           "Max Acumulado Estudiantes Desertados",
                           "Max Acumulado Horas Lectivas",
                           "Max Acumulado Numero Comentarios En Riesgo",
                           "last_max_idx_comentarios_en_riesgo",
                           #"last_max_idx_comentarios_en_mejora",
                           "last_max_idx_numero_estudiantes_per", "Max Acumulado Estudiantes Perdieron",
                           "last_max_idx_numero_estudiantes_deser", "last_max_idx_horas_lectivas",
                           "last_max_idx_estudiantes_aprobados",
                           "last_max_idx_puntaje_eval", "last_max_idx_puntaje_metod", "last_max_idx_puntaje_clar"], inplace=True)
    muestras.set_index(["id_docente"], inplace=True)
    return muestras

seguimiento_docente_en_cada_asignatura = (
    seguimiento_docente_en_cada_asignatura
    .set_index(["id_docente", "asignatura"], append=False)
    .groupby(level=["id_docente", "asignatura"], group_keys=False)
    .apply(agregar_metricas_docente_por_asignatura, include_groups=False)
    .reset_index()
)

seguimiento_docente_en_cada_asignatura = (
    seguimiento_docente_en_cada_asignatura
    .set_index(["id_docente", "asignatura"], append=False)
    .groupby(level=["id_docente", "asignatura"], group_keys=False)
    .apply(metricas_de_apoyo_para_detectar_reingreso, include_groups=False)
    .reset_index()
)

# seguimiento_docente_en_cada_asignatura["Reingreso"] = seguimiento_docente_en_cada_asignatura["Reingreso"].astype("Int64")

seguimiento_docente_en_cada_asignatura = seguimiento_docente_en_cada_asignatura.convert_dtypes().dropna(subset=["Comentarios"])

In [ ]:
seguimiento_docente_en_cada_asignatura

In [ ]:
corr_doc_asign = seguimiento_docente_en_cada_asignatura.select_dtypes(include="number").corr()

chosen_corrs_est = corr_doc_asign.loc[:,
    ["Estudiantes Aprobados"]
].sort_values(["Estudiantes Aprobados"], ascending=[False])[:9]

chosen_corrs_egr = corr_doc_asign.loc[:,
    ["Egresos a La Fecha"]
].sort_values(["Egresos a La Fecha"], ascending=[False])[:10]

In [ ]:
sns.heatmap(chosen_corrs_egr, annot=True)

In [ ]:
sns.heatmap(chosen_corrs_est, annot=True)

In [ ]:
# Entrena con variable target "Egresos a La Fecha" modelos de machine learning con el frame seguimiento_docente_en_cada_asignatura

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# Preparar datos para modelado
df_ml = seguimiento_docente_en_cada_asignatura.copy()

# Seleccionar solo columnas numéricas y eliminar NaNs
numeric_cols = df_ml.select_dtypes(include=['number']).columns.tolist()

# Remover la variable target y columnas no útiles
cols_to_remove = ['Egresos a La Fecha', 'Reingreso']  # Reingreso es booleano
feature_cols = [col for col in numeric_cols if col not in cols_to_remove]

# Preparar X e y
X = df_ml[feature_cols].dropna()
y = df_ml.loc[X.index, 'Egresos a La Fecha']

# Eliminar filas donde y es NaN
mask = ~y.isna()
X = X[mask]
y = y[mask]

print(f"Dataset shape: {X.shape}")
print(f"Features: {feature_cols}")

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Diccionario para almacenar resultados
results = {}

# 1. Ridge Regression
print("\n" + "=" * 50)
print("Training Ridge Regression...")
ridge_params = {
    'alpha': [0.1, 1.0, 10.0, 100.0],
    'solver': ['auto', 'svd', 'lsqr']
}
ridge = GridSearchCV(Ridge(), ridge_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
ridge.fit(X_train_scaled, y_train)
results['Ridge'] = {
    'model': ridge.best_estimator_,
    'params': ridge.best_params_,
    'train_score': ridge.best_score_,
    'y_pred': ridge.predict(X_test_scaled)
}
print(f"Best params: {ridge.best_params_}")

# 2. Lasso Regression
print("\n" + "=" * 50)
print("Training Lasso Regression...")
lasso_params = {
    'alpha': [0.01, 0.1, 1.0, 10.0],
    'max_iter': [1000, 5000]
}
lasso = GridSearchCV(Lasso(), lasso_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
lasso.fit(X_train_scaled, y_train)
results['Lasso'] = {
    'model': lasso.best_estimator_,
    'params': lasso.best_params_,
    'train_score': lasso.best_score_,
    'y_pred': lasso.predict(X_test_scaled)
}
print(f"Best params: {lasso.best_params_}")

# 3. ElasticNet
print("\n" + "=" * 50)
print("Training ElasticNet...")
elastic_params = {
    'alpha': [0.1, 1.0, 10.0],
    'l1_ratio': [0.2, 0.5, 0.8]
}
elastic = GridSearchCV(ElasticNet(), elastic_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
elastic.fit(X_train_scaled, y_train)
results['ElasticNet'] = {
    'model': elastic.best_estimator_,
    'params': elastic.best_params_,
    'train_score': elastic.best_score_,
    'y_pred': elastic.predict(X_test_scaled)
}
print(f"Best params: {elastic.best_params_}")

# 4. Decision Tree
print("\n" + "=" * 50)
print("Training Decision Tree...")
tree_params = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
tree = GridSearchCV(DecisionTreeRegressor(random_state=42), tree_params, cv=5, scoring='neg_mean_squared_error',
                    n_jobs=-1)
tree.fit(X_train, y_train)
results['DecisionTree'] = {
    'model': tree.best_estimator_,
    'params': tree.best_params_,
    'train_score': tree.best_score_,
    'y_pred': tree.predict(X_test)
}
print(f"Best params: {tree.best_params_}")

# 5. Random Forest
print("\n" + "=" * 50)
print("Training Random Forest...")
rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}
rf = GridSearchCV(RandomForestRegressor(random_state=42), rf_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
rf.fit(X_train, y_train)
results['RandomForest'] = {
    'model': rf.best_estimator_,
    'params': rf.best_params_,
    'train_score': rf.best_score_,
    'y_pred': rf.predict(X_test)
}
print(f"Best params: {rf.best_params_}")

# 6. Gradient Boosting
print("\n" + "=" * 50)
print("Training Gradient Boosting...")
gb_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}
gb = GridSearchCV(GradientBoostingRegressor(random_state=42), gb_params, cv=5, scoring='neg_mean_squared_error',
                  n_jobs=-1)
gb.fit(X_train, y_train)
results['GradientBoosting'] = {
    'model': gb.best_estimator_,
    'params': gb.best_params_,
    'train_score': gb.best_score_,
    'y_pred': gb.predict(X_test)
}
print(f"Best params: {gb.best_params_}")

# 7. K-Nearest Neighbors
print("\n" + "=" * 50)
print("Training K-Nearest Neighbors...")
knn_params = {
    'n_neighbors': [3, 5, 7, 10],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}
knn = GridSearchCV(KNeighborsRegressor(), knn_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
knn.fit(X_train_scaled, y_train)
results['KNN'] = {
    'model': knn.best_estimator_,
    'params': knn.best_params_,
    'train_score': knn.best_score_,
    'y_pred': knn.predict(X_test_scaled)
}
print(f"Best params: {knn.best_params_}")

# 8. SVR
print("\n" + "=" * 50)
print("Training Support Vector Regression...")
svr_params = {
    'kernel': ['linear', 'rbf'],
    'C': [0.1, 1, 10],
    'epsilon': [0.01, 0.1, 0.2]
}
svr = GridSearchCV(SVR(), svr_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
svr.fit(X_train_scaled, y_train)
results['SVR'] = {
    'model': svr.best_estimator_,
    'params': svr.best_params_,
    'train_score': svr.best_score_,
    'y_pred': svr.predict(X_test_scaled)
}
print(f"Best params: {svr.best_params_}")

# Evaluar todos los modelos
print("\n" + "=" * 70)
print("EVALUATION RESULTS - Target: Egresos a La Fecha")
print("=" * 70)

comparison_df = pd.DataFrame()

for name, result in results.items():
    y_pred = result['y_pred']

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    comparison_df = pd.concat([comparison_df, pd.DataFrame({
        'Model': [name],
        'RMSE': [rmse],
        'MAE': [mae],
        'R2': [r2],
        'Best_Params': [str(result['params'])]
    })], ignore_index=True)

    print(f"\n{name}:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  R2:   {r2:.4f}")
    print(f"  Best params: {result['params']}")

# Ordenar por R2 score
comparison_df = comparison_df.sort_values('R2', ascending=False)

print("\n" + "=" * 70)
print("RANKING (by R2 score):")
print("=" * 70)
print(comparison_df.to_string(index=False))

# Guardar el mejor modelo
best_model_name = comparison_df.iloc[0]['Model']
best_model = results[best_model_name]['model']

print("\n" + "=" * 70)
print(f"BEST MODEL: {best_model_name}")
print(f"Best Parameters: {results[best_model_name]['params']}")
print(f"R2 Score: {comparison_df.iloc[0]['R2']:.4f}")
print("=" * 70)


In [23]:
# Entrena con variable target "Estudiantes Aprobados" modelos de machine learning con el frame seguimiento_docente_en_cada_asignatura

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# Preparar datos para modelado
df_ml = seguimiento_docente_en_cada_asignatura.copy()

# Seleccionar solo columnas numéricas y eliminar NaNs
numeric_cols = df_ml.select_dtypes(include=['number']).columns.tolist()

# Remover la variable target y columnas no útiles
cols_to_remove = ['Estudiantes Aprobados', 'Reingreso']  # Reingreso es booleano
feature_cols = [col for col in numeric_cols if col not in cols_to_remove]

# Preparar X e y
X = df_ml[feature_cols].dropna()
y = df_ml.loc[X.index, 'Estudiantes Aprobados']

# Eliminar filas donde y es NaN
mask = ~y.isna()
X = X[mask]
y = y[mask]

print(f"Dataset shape: {X.shape}")
print(f"Features: {feature_cols}")

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Diccionario para almacenar resultados
results = {}

# 1. Ridge Regression
print("\n" + "=" * 50)
print("Training Ridge Regression...")
ridge_params = {
    'alpha': [0.1, 1.0, 10.0, 100.0],
    'solver': ['auto', 'svd', 'lsqr']
}
ridge = GridSearchCV(Ridge(), ridge_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
ridge.fit(X_train_scaled, y_train)
results['Ridge'] = {
    'model': ridge.best_estimator_,
    'params': ridge.best_params_,
    'train_score': ridge.best_score_,
    'y_pred': ridge.predict(X_test_scaled)
}
print(f"Best params: {ridge.best_params_}")

# 2. Lasso Regression
print("\n" + "=" * 50)
print("Training Lasso Regression...")
lasso_params = {
    'alpha': [0.01, 0.1, 1.0, 10.0],
    'max_iter': [1000, 5000]
}
lasso = GridSearchCV(Lasso(), lasso_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
lasso.fit(X_train_scaled, y_train)
results['Lasso'] = {
    'model': lasso.best_estimator_,
    'params': lasso.best_params_,
    'train_score': lasso.best_score_,
    'y_pred': lasso.predict(X_test_scaled)
}
print(f"Best params: {lasso.best_params_}")

# 3. ElasticNet
print("\n" + "=" * 50)
print("Training ElasticNet...")
elastic_params = {
    'alpha': [0.1, 1.0, 10.0],
    'l1_ratio': [0.2, 0.5, 0.8]
}
elastic = GridSearchCV(ElasticNet(), elastic_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
elastic.fit(X_train_scaled, y_train)
results['ElasticNet'] = {
    'model': elastic.best_estimator_,
    'params': elastic.best_params_,
    'train_score': elastic.best_score_,
    'y_pred': elastic.predict(X_test_scaled)
}
print(f"Best params: {elastic.best_params_}")

# 4. Decision Tree
print("\n" + "=" * 50)
print("Training Decision Tree...")
tree_params = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
tree = GridSearchCV(DecisionTreeRegressor(random_state=42), tree_params, cv=5, scoring='neg_mean_squared_error',
                    n_jobs=-1)
tree.fit(X_train, y_train)
results['DecisionTree'] = {
    'model': tree.best_estimator_,
    'params': tree.best_params_,
    'train_score': tree.best_score_,
    'y_pred': tree.predict(X_test)
}
print(f"Best params: {tree.best_params_}")

# 5. Random Forest
print("\n" + "=" * 50)
print("Training Random Forest...")
rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}
rf = GridSearchCV(RandomForestRegressor(random_state=42), rf_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
rf.fit(X_train, y_train)
results['RandomForest'] = {
    'model': rf.best_estimator_,
    'params': rf.best_params_,
    'train_score': rf.best_score_,
    'y_pred': rf.predict(X_test)
}
print(f"Best params: {rf.best_params_}")

# 6. Gradient Boosting
print("\n" + "=" * 50)
print("Training Gradient Boosting...")
gb_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}
gb = GridSearchCV(GradientBoostingRegressor(random_state=42), gb_params, cv=5, scoring='neg_mean_squared_error',
                  n_jobs=-1)
gb.fit(X_train, y_train)
results['GradientBoosting'] = {
    'model': gb.best_estimator_,
    'params': gb.best_params_,
    'train_score': gb.best_score_,
    'y_pred': gb.predict(X_test)
}
print(f"Best params: {gb.best_params_}")

# 7. K-Nearest Neighbors
print("\n" + "=" * 50)
print("Training K-Nearest Neighbors...")
knn_params = {
    'n_neighbors': [3, 5, 7, 10],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}
knn = GridSearchCV(KNeighborsRegressor(), knn_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
knn.fit(X_train_scaled, y_train)
results['KNN'] = {
    'model': knn.best_estimator_,
    'params': knn.best_params_,
    'train_score': knn.best_score_,
    'y_pred': knn.predict(X_test_scaled)
}
print(f"Best params: {knn.best_params_}")

# 8. SVR
print("\n" + "=" * 50)
print("Training Support Vector Regression...")
svr_params = {
    'kernel': ['linear', 'rbf'],
    'C': [0.1, 1, 10],
    'epsilon': [0.01, 0.1, 0.2]
}
svr = GridSearchCV(SVR(), svr_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
svr.fit(X_train_scaled, y_train)
results['SVR'] = {
    'model': svr.best_estimator_,
    'params': svr.best_params_,
    'train_score': svr.best_score_,
    'y_pred': svr.predict(X_test_scaled)
}
print(f"Best params: {svr.best_params_}")

# Evaluar todos los modelos
print("\n" + "=" * 70)
print("EVALUATION RESULTS - Target: Estudiantes Aprobados")
print("=" * 70)

comparison_df = pd.DataFrame()

for name, result in results.items():
    y_pred = result['y_pred']

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    comparison_df = pd.concat([comparison_df, pd.DataFrame({
        'Model': [name],
        'RMSE': [rmse],
        'MAE': [mae],
        'R2': [r2],
        'Best_Params': [str(result['params'])]
    })], ignore_index=True)

    print(f"\n{name}:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  R2:   {r2:.4f}")
    print(f"  Best params: {result['params']}")

# Ordenar por R2 score
comparison_df = comparison_df.sort_values('R2', ascending=False)

print("\n" + "=" * 70)
print("RANKING (by R2 score):")
print("=" * 70)
print(comparison_df.to_string(index=False))

# Guardar el mejor modelo
best_model_name = comparison_df.iloc[0]['Model']
best_model = results[best_model_name]['model']

print("\n" + "=" * 70)
print(f"BEST MODEL: {best_model_name}")
print(f"Best Parameters: {results[best_model_name]['params']}")
print(f"R2 Score: {comparison_df.iloc[0]['R2']:.4f}")
print("=" * 70)

Dataset shape: (1246, 30)
Features: ['Cantidad de Grupos', 'numero_estudiantes', 'puntaje_claridad', 'puntaje_metodologia', 'puntaje_evaluacion', 'Horas Lectivas', 'Estudiantes Desertados', 'Estudiantes Perdieron', 'Numero Comentarios Estables', 'Numero Comentarios En Riesgo', 'Numero Comentarios En Mejora', 'Semestres Desde Ultima Calificación', 'Indice de Reingreso', 'Egresos a La Fecha', 'Semestres Sin Incidente Comentarios Negativos', 'Ultimo Incidente Comentarios Negativos', 'Semestres Sin Mejora en Estudiantes Aprobados', 'Ultimo Hito Estudiantes Aprobados', 'Semestres Sin Mejora en Puntaje Evaluación', 'Ultimo Hito Puntaje Evaluación', 'Semestres Sin Mejora en Puntaje Metodología', 'Ultimo Hito Puntaje Metodología', 'Semestres Sin Mejora en Puntaje Claridad', 'Ultimo Hito Puntaje Claridad', 'Semestres Sin Exceso en Horas Lectivas', 'Ultimo Tope Horas Lectivas', 'Semestres Sin Incidente Estudiantes Desertados', 'Ultimo Incidente Estudiantes Desertados', 'Semestres Sin Incidente E

In [ ]:
seguimiento_asignaturas = (
    df.groupby(["asignatura", "semestre"], as_index=False)
    .agg(
        Cantidad_de_Docentes=("id_docente", "nunique"),
        Cantidad_de_Grupos=("semestre", "size"), # Counts rows per group
        numero_estudiantes=("numero_estudiantes", "sum"),
        Asistencia_Tipica_Estudiantes=("promedio_asistencias_estudiantes", "sum"),
        Horas_Lectivas=("Horas Lectivas Totales", "sum"),
        Estudiantes_Aprobados=("numero_estudiantes_aprobaron", "sum"),
        Estudiantes_Perdieron=("numero_estudiantes_perdieron", "sum"),
        Estudiantes_Desistieron=("numero_estudiantes_desistieron", "sum"),
        Clases_Dictadas=("clases_dictadas", "sum"),
        # Franja_Horaria=("franja_horaria", lambda x: "Mixta" if len(x.dropna().unique()) > 1 else x[0]),
    )
    .rename(columns={
        "Cantidad_de_Grupos": "Cantidad de Grupos",
        "Cantidad_de_Docentes": "Cantidad de Docentes",
        "Asistencia_Tipica_Estudiantes": "Asistencia Típica de Estudiantes",
        "Horas_Lectivas": "Horas Lectivas",
        "Estudiantes_Aprobados": "Estudiantes Aprobados",
        "Estudiantes_Perdieron": "Estudiantes Que Perdieron",
        "Estudiantes_Desistieron": "Estudiantes Que Desistieron",
        "Clases_Dictadas": "Clases Dictadas",
        # "Franja_Horaria": "Franja Horaria",
    })
    .sort_values(by=["asignatura", "semestre"], ascending=[True, True])
)

seguimiento_asignaturas["Cantidad de Docentes"] = seguimiento_asignaturas["Cantidad de Docentes"].astype("Int64")
seguimiento_asignaturas["Cantidad de Grupos"] = seguimiento_asignaturas["Cantidad de Grupos"].astype("Int64")
seguimiento_asignaturas["numero_estudiantes"] = seguimiento_asignaturas["numero_estudiantes"].astype("Int64")

seguimiento_asignaturas["Asistencia Típica de Estudiantes"] = seguimiento_asignaturas["Asistencia Típica de Estudiantes"].astype("Float64")
seguimiento_asignaturas["Horas Lectivas"] = seguimiento_asignaturas["Horas Lectivas"].astype("Int64")
seguimiento_asignaturas["Estudiantes Aprobados"] = seguimiento_asignaturas["Estudiantes Aprobados"].astype("Int64")
seguimiento_asignaturas["Clases Dictadas"] = seguimiento_asignaturas["Clases Dictadas"].astype("Int64")

# seguimiento_asignaturas["Franja Horaria"] = seguimiento_asignaturas["Franja Horaria"].astype("category")

seguimiento_asignaturas = seguimiento_asignaturas.convert_dtypes()
seguimiento_asignaturas


In [ ]:
corr_asign = seguimiento_asignaturas.select_dtypes(include='number').corr()

In [ ]:
sns.heatmap(corr_asign.loc[:, ["Estudiantes Aprobados"]].sort_values(["Estudiantes Aprobados"], ascending=False), annot=True)

In [ ]:
sns.heatmap(corr_asign.loc[:, ["Estudiantes Que Desistieron"]].sort_values(["Estudiantes Que Desistieron"], ascending=False), annot=True)

In [ ]:
sns.heatmap(corr_asign.loc[:, ["Estudiantes Que Perdieron"]].sort_values(["Estudiantes Que Perdieron"], ascending=False), annot=True)